In [ ]:
import sys

sys.path.append("../..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster._supervised import pair_confusion_matrix

from wildlife_datasets.datasets.utils import find_images

In [ ]:
def score(solution: pd.DataFrame, submission: pd.DataFrame) -> dict:
    submission = submission.reset_index(drop=True)
    solution = solution.reset_index(drop=True)

    # Checks for correct input format
    if "image_id" not in submission.columns:
        return {}
    if "cluster" not in submission.columns:
        return {}
    if len(submission) != len(solution):
        return {}
    if not np.array_equal(submission["image_id"], solution["image_id"]):
        return {}
    if not pd.api.types.is_string_dtype(submission["cluster"]):
        return {}
    prefixes = (
        "cluster_LynxID2025_",
        "cluster_SalamanderID2025_",
        "cluster_SeaTurtleID2022_",
        "cluster_TexasHornedLizards_",
    )
    if not submission["cluster"].str.startswith(prefixes).all():
        return {}

    # Compute adjusted random index (ARI) for each dataset
    results = {}
    for name, solution_dataset in solution.groupby("dataset"):
        predictions = submission.loc[solution_dataset.index, "cluster"].to_numpy()
        labels = solution_dataset["identity"].to_numpy()
        results[name] = adjusted_rand_score(labels, predictions)

    # Add the mean score
    results["mean"] = np.mean(list(results.values()))

    return results


def count_clusters(df):
    df = df.copy()
    df["dataset"] = df["cluster"].apply(lambda x: x.split("_")[1])

    n_clusters = {}
    for dataset, df_dataset in df.groupby("dataset"):
        n_clusters[dataset] = df_dataset["cluster"].nunique()
    return n_clusters


def extract_participant_info(participant, participants_replace={}):
    participant_split = participant.split(" ")
    participant_rank = int(participant_split[0])
    participant_name = " ".join(participant_split[1:])
    if participant_name in participants_replace:
        participant_name = participants_replace[participant_name]
    return participant_name, participant_rank


def get_submission(i, results, submissions):
    i_submission = results["submission"].iloc[i]
    file_name = submissions.loc[i_submission, "file_name"]
    return pd.read_csv(file_name)

In [ ]:
root = "/data/wildlife_datasets/data/AnimalCLEF2026"
root_submission = f"{root}/submissions"

participants_replace = {
    "dikdagaio8psovvcom": "dikdagaio8@psovv.com",
    "DSGT LifeCLEF": "DS@GT LifeCLEF",
    "maatuinnf0psovvcom": "maatuinnf0@psovv.com",
    "niftudg1drcom": "niftudg1dr.com",
    "MIA": "M.I.A",
    "Noahs Turtle Ship": "Noah's Turtle Ship",
    "SCNUDT": "SCNU-DT",
    "VIPLVSU": "VIPL-VSU",
    "vSXQBztDRW8De": "vSXQBz.tDR!W8De",
    "woorirrnl6xghffcom": "woorirrnl6@xghff.com",
}

In [ ]:
solution = pd.read_csv(f"{root}/solution.csv")
solution["cluster"] = "cluster_" + solution["identity"]

scores = pd.read_csv(f"{root}/scores.csv")
teams = scores["TeamName"].unique().tolist()

idx_public = solution["Usage"] == "Public"
idx_private = ~idx_public
submissions = find_images(root_submission, img_extensions=(".csv",))
n_participants = submissions["path"].nunique()

In [ ]:
mask = submissions["path"].str.contains("/")
if mask.any():
    display(submissions[mask])
    raise ValueError("There are multiple folders in the submission.")

In [ ]:
emails = scores["UserEmail"].unique().tolist()
emails = sorted([x for x in emails if "@" in x])
# ", ".join(emails)

In [ ]:
scores_public = []
scores_private = []
for i, submission in submissions.iterrows():
    participant_name, participant_rank = extract_participant_info(
        submission["path"], participants_replace=participants_replace
    )
    submissions.loc[i, "participant"] = participant_name
    submissions.loc[i, "participant_rank"] = participant_rank

    file_name = f"{root_submission}/{submission['path']}/{submission['file']}"
    submissions.loc[i, "file_name"] = file_name

    try:
        submission = pd.read_csv(file_name)
    except Exception:
        print(f"Loading failed: {file_name}")
        submission = pd.DataFrame()

    if solution.index.equals(submission.index):
        score_public = score(solution[idx_public], submission[idx_public])
        score_private = score(solution[idx_private], submission[idx_private])
    else:
        score_public = {}
        score_private = {}

    scores_public.append(score_public)
    scores_private.append(score_private)

submissions["participant_rank"] = submissions["participant_rank"].astype(int)
public = pd.DataFrame(scores_public)
private = pd.DataFrame(scores_private)

In [ ]:
(private == 1).sum()

In [ ]:
submissions.loc[private["TexasHornedLizards"] == 1, "participant"].unique()

In [ ]:
results = {}
for participant, df_participant in submissions.groupby("participant"):
    rank = df_participant["participant_rank"].iloc[0]

    idx = df_participant.index.to_list()
    i_max = public.loc[idx, "mean"].argmax()
    i_submission = idx[i_max]
    results[participant] = private.iloc[i_submission]
    results[participant]["rank"] = rank
    results[participant]["submission"] = i_submission

results = pd.DataFrame(results).T
results["rank"] = results["rank"].astype(int)
results["submission"] = results["submission"].astype(int)
results = results.sort_values("rank")

In [ ]:
x_min = 1
x_max = len(results)
x_range = range(x_min, x_max + 1)

plot_names = {
    "LynxID2025": "lynxes",
    "SalamanderID2025": "salamanders",
    "SeaTurtleID2022": "sea turtles",
    "TexasHornedLizards": "lizards",
    "mean": "mean"
}

plt.figure(figsize=(14, 2.5))
for i, (col, plot_name) in enumerate(plot_names.items()):
    plt.subplot(1, len(plot_names), i + 1)
    plt.plot(x_range, results[col])
    plt.hlines(results[col].mean(), x_min, x_max, linestyle="--", color="orange")
    plt.title(plot_names[col])
    plt.xlabel("team rank")
    if i == 0:
        plt.ylabel("ARI")
    plt.xlim([0, 50])
    plt.ylim([0, 1])
    plt.grid(alpha=0.4)
plt.savefig("ari.png", dpi=300, bbox_inches="tight")

In [ ]:
n_clusters_dict = []
for i in range(len(results)):
    submission = get_submission(i, results, submissions)
    n_clusters_dict.append(count_clusters(submission))

n_clusters = pd.DataFrame(n_clusters_dict)
n_clusters_true = count_clusters(solution)

In [ ]:
1.5*7, 1.5*3.8

In [ ]:
cols = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022", "TexasHornedLizards"]

plt.figure(figsize=(10, 5.5))
plt.subplots_adjust(wspace=0.15, hspace=0.3)
for i, col in enumerate(cols):
    true = n_clusters_true[col]
    plt.subplot(2, 2, i + 1)
    plt.plot(range(1, 51), n_clusters[col])
    plt.hlines(true, x_min, x_max, color="orange")
    if col in ["LynxID2025", "SeaTurtleID2022"]:
        plt.hlines(2 * true, x_min, x_max, linestyle="--", color="orange")
    plt.title(plot_names[col])
    if i in (2, 3):
        plt.xlabel("team rank")
    if i in (0, 2):
        plt.ylabel("population estimate")
    plt.xlim([0, 50])
    plt.grid(alpha=0.4)

plt.savefig("population.png", dpi=300, bbox_inches="tight")

In [ ]:
n_top = 5
y_true = solution["identity"]

data = np.zeros((0, 8))
dataset_names = solution["dataset"].unique()
n_clusters = {}
for i in range(n_top):
    submission = get_submission(i, results, submissions)
    team_name = results.index[i]
    y_pred = submission["cluster"]

    data_part = np.zeros((2, 0))
    n_clusters_part = {}
    for dataset_name in dataset_names:
        mask = solution["dataset"] == dataset_name
        confusion = pair_confusion_matrix(y_true[mask], y_pred[mask])

        data_part = np.hstack((data_part, confusion))
        n_clusters_part[plot_names[dataset_name]] = y_pred[mask].nunique()
    data = np.vstack((data, data_part))
    n_clusters[team_name] = n_clusters_part

n_clusters_part = {}
for dataset_name in dataset_names:
    mask = solution["dataset"] == dataset_name
    n_clusters_part[plot_names[dataset_name]] = y_true[mask].nunique()

n_clusters["TRUE"] = n_clusters_part

In [ ]:
data = pd.DataFrame(data)
for j in data.columns:
    data[j] = data[j].astype(int).astype(str)
for j in data.columns:
    for i in range(len(data)):
        prefix = "\\colorA" if np.mod(i // 2 + j // 2, 2) == 0 else "\\colorB"
        data.loc[i, j] = prefix + " " + data.loc[i, j]

print(data.to_latex(float_format="%.3f"))

In [ ]:
print(pd.DataFrame(n_clusters).T.to_latex())